In [ ]:
def main(datasources, start_date, end_date):
    """
    R04: rawpool_family_fix_mean_4d

    R01 rawpool family-fixed distilled factor with the same four-day
    persistence layer that turned F18-v4 into F37:
    1) build the R01 raw/basic expanded candidate set,
    2) clean candidates cross-sectionally with industry neutralization,
    3) combine the R01 family-fixed static weights,
    4) apply a final 4-day instrument-level rolling mean and re-rank by date.
    """
    import numpy as np
    import pandas as pd
    import dai

    datasources = datasources or {}
    bar1m_table = datasources.get("bar1m", "bigalpha_2026_stock_bar1m")
    financial_table = datasources.get("financial", "bigalpha_2026_financial")
    factorlib_table = datasources.get("factorlib", "bigalpha_2026_factorlib")
    exposure_table = datasources.get("exposure", "bigalpha_2026_exposure")

    start_ts = pd.to_datetime(start_date)
    end_ts = pd.to_datetime(end_date)
    query_start = max(pd.Timestamp("2019-01-01"), start_ts - pd.Timedelta(days=50))
    query_end = end_ts + pd.Timedelta(days=1)

    feature_cols = [
        "F01",
        "F02",
        "F03",
        "F08",
        "F14",
        "F17",
        "depth_slope_asymmetry",
        "spread_tightness",
        "depth_resilience",
        "trade_exhaustion",
        "average_trade_size",
        "intraday_reversal",
        "close_location",
        "opening_gap_reversal",
        "opening_absorption",
        "closing_pressure_shift",
        "pressure_instability",
        "amount_concentration",
        "profitability_quality",
        "fundamental_growth",
        "fundamental_value",
        "balance_sheet_safety",
        "cashflow_quality",
        "accrual_quality",
        "asset_turnover",
        "rd_intensity",
        "working_capital_safety",
        "short_reversal_5",
        "medium_momentum_20",
        "realized_volatility_20",
        "amount_shock_20",
        "amihud_20",
    ]

    expanded_cols = [
        "spread_pressure_quality",
        "closing_pressure_stability",
        "opening_depth_recovery",
        "style_residual_pressure",
        "depth_recovery_after_range",
        "liquidity_vol_balance",
        "quality_value_micro",
    ]

    rawpool_cols = [
        "range_mean_5d",
        "price_to_low_3d",
        "amount_mean_3d",
        "illiq_3d",
    ]

    # Distilled from f18_v4_rawpool_step1_family_fix_final_weights.csv.
    # Same R01 weights; the only R04 change is final 4-day persistence.
    final_weights = {
        "F02": 0.548623000000,
        "range_mean_5d": -0.098235000000,
        "amount_shock_20": -0.071683000000,
        "price_to_low_3d": -0.065292000000,
        "amount_mean_3d": -0.049498000000,
        "close_location": -0.046880000000,
        "intraday_reversal": 0.027618000000,
        "realized_volatility_20": -0.023902000000,
        "F17": 0.019953000000,
        "medium_momentum_20": -0.013428000000,
        "short_reversal_5": 0.011066000000,
        "illiq_3d": 0.008838000000,
        "trade_exhaustion": 0.008081000000,
        "pressure_instability": -0.006903000000,
    }

    sql = f"""
    WITH minute_features AS (
        SELECT
            date,
            instrument,
            open,
            high,
            low,
            close,
            pre_close,
            volume,
            amount,
            deal_number,
            (
                COALESCE(bid_volume1, 0) + COALESCE(bid_volume2, 0) + COALESCE(bid_volume3, 0)
                - COALESCE(ask_volume1, 0) - COALESCE(ask_volume2, 0) - COALESCE(ask_volume3, 0)
            ) / (
                COALESCE(bid_volume1, 0) + COALESCE(bid_volume2, 0) + COALESCE(bid_volume3, 0)
                + COALESCE(ask_volume1, 0) + COALESCE(ask_volume2, 0) + COALESCE(ask_volume3, 0)
                + 1.0
            ) AS pressure3,
            (
                COALESCE(bid_volume1, 0) + COALESCE(bid_volume2, 0) + COALESCE(bid_volume3, 0)
                + COALESCE(bid_volume4, 0) + COALESCE(bid_volume5, 0)
                - COALESCE(ask_volume1, 0) - COALESCE(ask_volume2, 0) - COALESCE(ask_volume3, 0)
                - COALESCE(ask_volume4, 0) - COALESCE(ask_volume5, 0)
            ) / (
                COALESCE(bid_volume1, 0) + COALESCE(bid_volume2, 0) + COALESCE(bid_volume3, 0)
                + COALESCE(bid_volume4, 0) + COALESCE(bid_volume5, 0)
                + COALESCE(ask_volume1, 0) + COALESCE(ask_volume2, 0) + COALESCE(ask_volume3, 0)
                + COALESCE(ask_volume4, 0) + COALESCE(ask_volume5, 0) + 1.0
            ) AS pressure5,
            COALESCE(
                (
                    COALESCE(ask_price1, close) * COALESCE(bid_volume1, 0)
                    + COALESCE(bid_price1, close) * COALESCE(ask_volume1, 0)
                ) / NULLIF(COALESCE(bid_volume1, 0) + COALESCE(ask_volume1, 0), 0),
                close
            ) AS microprice,
            (
                COALESCE(bid_volume1, 0) + COALESCE(bid_volume2, 0) + COALESCE(bid_volume3, 0)
            ) / (
                COALESCE(bid_num_orders1, 0) + COALESCE(bid_num_orders2, 0)
                + COALESCE(bid_num_orders3, 0) + 1.0
            ) AS bid_size_per_order,
            (
                COALESCE(ask_volume1, 0) + COALESCE(ask_volume2, 0) + COALESCE(ask_volume3, 0)
            ) / (
                COALESCE(ask_num_orders1, 0) + COALESCE(ask_num_orders2, 0)
                + COALESCE(ask_num_orders3, 0) + 1.0
            ) AS ask_size_per_order,
            (
                COALESCE(bid_num_orders1, 0) + COALESCE(bid_num_orders2, 0)
                + COALESCE(bid_num_orders3, 0) + COALESCE(ask_num_orders1, 0)
                + COALESCE(ask_num_orders2, 0) + COALESCE(ask_num_orders3, 0)
            ) AS total_order_count3,
            (
                COALESCE(bid_volume1, 0) + COALESCE(bid_volume2, 0) + COALESCE(bid_volume3, 0)
                + COALESCE(ask_volume1, 0) + COALESCE(ask_volume2, 0) + COALESCE(ask_volume3, 0)
            ) AS depth3,
            (
                COALESCE(bid_volume1, 0) + COALESCE(bid_volume2, 0) + COALESCE(bid_volume3, 0)
            ) AS bid_depth3,
            (
                COALESCE(ask_volume1, 0) + COALESCE(ask_volume2, 0) + COALESCE(ask_volume3, 0)
            ) AS ask_depth3,
            (COALESCE(bid_volume4, 0) + COALESCE(bid_volume5, 0)) AS bid_depth45,
            (COALESCE(ask_volume4, 0) + COALESCE(ask_volume5, 0)) AS ask_depth45,
            (COALESCE(ask_price1, close) - COALESCE(bid_price1, close))
                / NULLIF(close, 0) AS spread_ratio,
            EXTRACT(HOUR FROM date) * 60 + EXTRACT(MINUTE FROM date) AS minute_of_day
        FROM {bar1m_table}
    ),
    daily AS (
        SELECT
            date_trunc('day', date)::DATE AS date,
            instrument,
            ARG_MIN(open, date) AS open,
            ARG_MAX(close, date) AS close,
            MAX(pre_close) AS pre_close,
            MAX(high) AS day_high,
            MIN(low) AS day_low,
            AVG(pressure3) AS pressure3_mean,
            STDDEV_POP(pressure3) AS pressure3_std,
            AVG(ABS(pressure3)) AS pressure3_abs_mean,
            AVG(pressure5) AS pressure5_mean,
            AVG((microprice - close) / NULLIF(close, 0)) AS micro_gap,
            (MAX(high) - MIN(low)) / NULLIF(AVG(close), 0) AS range_ratio,
            (MAX(close) - MIN(close)) / NULLIF(AVG(close), 0) AS close_range,
            ARG_MAX(close, date) / NULLIF(AVG(close), 0) - 1.0 AS extension,
            SUM(amount) AS amount_sum,
            AVG(amount) AS amount_mean,
            STDDEV_POP(amount) AS amount_std,
            SUM(volume) AS volume_sum,
            SUM(deal_number) AS deal_number_sum,
            AVG(depth3) AS depth3_mean,
            AVG(bid_depth3) AS bid_depth3_mean,
            AVG(ask_depth3) AS ask_depth3_mean,
            AVG(bid_depth45) AS bid_depth45_mean,
            AVG(ask_depth45) AS ask_depth45_mean,
            AVG(spread_ratio) AS spread_ratio_mean,
            AVG(CASE WHEN minute_of_day <= 600 THEN pressure3 END) AS opening_pressure,
            AVG(CASE WHEN minute_of_day >= 870 THEN pressure3 END) AS closing_pressure,
            AVG(
                LN(1.0 + COALESCE(bid_size_per_order, 0.0))
                - LN(1.0 + COALESCE(ask_size_per_order, 0.0))
            ) AS fragmentation,
            STDDEV_POP(
                LN(1.0 + COALESCE(bid_size_per_order, 0.0))
                - LN(1.0 + COALESCE(ask_size_per_order, 0.0))
            ) AS frag_volatility,
            AVG(total_order_count3) AS order_count_mean,
            STDDEV_POP(total_order_count3) AS order_count_std
        FROM minute_features
        GROUP BY date_trunc('day', date)::DATE, instrument
    )
    SELECT * FROM daily ORDER BY date, instrument
    """

    raw = dai.query(sql, filters={"date": [query_start, query_end]}, compression=True).df()
    raw["date"] = pd.to_datetime(raw["date"])
    raw["instrument"] = raw["instrument"].astype(str)
    for col in [c for c in raw.columns if c not in ["date", "instrument"]]:
        raw[col] = pd.to_numeric(raw[col], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)

    def rank_z(series):
        ranked = series.rank(method="average", pct=True)
        centered = ranked - ranked.mean()
        scale = centered.std(ddof=0)
        if not np.isfinite(scale) or scale <= 1e-12:
            return centered * 0.0
        return centered / scale

    def winsorize_cross_section(series, lower=0.01, upper=0.99):
        finite = series.replace([np.inf, -np.inf], np.nan)
        if finite.notna().sum() < 20:
            return finite
        return finite.clip(lower=finite.quantile(lower), upper=finite.quantile(upper))

    financial_start = max(pd.Timestamp("2019-01-01"), query_start - pd.Timedelta(days=550))
    financial = dai.query(
        f"""
        SELECT
            date,
            instrument,
            report_date,
            net_cffoa,
            net_profit,
            total_assets,
            total_liabilities,
            total_equity_to_parent_shareholders,
            latest_shares,
            operating_revenue,
            research_and_development_expense,
            total_current_assets,
            total_current_liabilities
        FROM {financial_table}
        WHERE category = 'lf' AND shift = 0
        """,
        filters={"date": [financial_start, query_end]},
        compression=True,
    ).df()
    pit_cols = [
        "cashflow_quality",
        "accrual_quality",
        "asset_turnover",
        "rd_intensity",
        "working_capital_safety",
        "balance_sheet_safety",
        "book_value_per_share",
        "fundamental_growth",
    ]
    if financial.empty:
        for col in pit_cols:
            raw[col] = 0.0
    else:
        financial["date"] = pd.to_datetime(financial["date"])
        financial["report_date"] = pd.to_datetime(financial["report_date"])
        financial["instrument"] = financial["instrument"].astype(str)
        financial_inputs = [
            "net_cffoa",
            "net_profit",
            "total_assets",
            "total_liabilities",
            "total_equity_to_parent_shareholders",
            "latest_shares",
            "operating_revenue",
            "research_and_development_expense",
            "total_current_assets",
            "total_current_liabilities",
        ]
        for col in financial_inputs:
            financial[col] = pd.to_numeric(financial[col], errors="coerce").replace([np.inf, -np.inf], np.nan)
        assets = financial["total_assets"].abs().replace(0.0, np.nan)
        revenue = financial["operating_revenue"].abs().replace(0.0, np.nan)
        profit_scale = financial["net_profit"].abs().replace(0.0, np.nan)
        financial["cashflow_quality"] = financial["net_cffoa"] / profit_scale
        financial["accrual_quality"] = -(financial["net_profit"] - financial["net_cffoa"]) / assets
        financial["asset_turnover"] = financial["operating_revenue"] / assets
        financial["rd_intensity"] = financial["research_and_development_expense"] / revenue
        financial["working_capital_safety"] = (
            financial["total_current_assets"] - financial["total_current_liabilities"]
        ) / assets
        financial["balance_sheet_safety"] = -financial["total_liabilities"] / assets
        financial["book_value_per_share"] = (
            financial["total_equity_to_parent_shareholders"]
            / financial["latest_shares"].abs().replace(0.0, np.nan)
        )
        report_values = (
            financial[["instrument", "report_date", "operating_revenue"]]
            .dropna(subset=["report_date"])
            .drop_duplicates(["instrument", "report_date"], keep="last")
        )
        prior_values = report_values.rename(columns={"operating_revenue": "prior_year_revenue"}).copy()
        prior_values["report_date"] = prior_values["report_date"] + pd.DateOffset(years=1)
        financial = pd.merge(financial, prior_values, how="left", on=["instrument", "report_date"])
        financial["fundamental_growth"] = (
            financial["operating_revenue"] / financial["prior_year_revenue"].abs().replace(0.0, np.nan) - 1.0
        )
        financial = (
            financial[["date", "instrument"] + pit_cols]
            .drop_duplicates(["date", "instrument"], keep="last")
            .sort_values(["date", "instrument"])
        )
        raw = pd.merge_asof(
            raw.sort_values(["date", "instrument"]),
            financial,
            on="date",
            by="instrument",
            direction="backward",
            allow_exact_matches=True,
        )
        for col in pit_cols:
            raw[col] = pd.to_numeric(raw[col], errors="coerce").replace([np.inf, -np.inf], np.nan)
    raw["fundamental_value"] = raw["book_value_per_share"] / raw["close"].replace(0.0, np.nan)

    quality = dai.query(
        f"""
        SELECT
            date::DATE AS date,
            instrument,
            COALESCE(roa_avg_ttm, 0.0)
            + COALESCE(gross_profit_rate_ttm, 0.0)
            - 0.35 * COALESCE(debt_to_asset_lf, 0.0) AS quality_score,
            0.10 * COALESCE(current_ratio_lf, 0.0) AS liquidity_quality
        FROM {factorlib_table}
        """,
        filters={"date": [query_start, query_end]},
        compression=True,
    ).df()
    if quality.empty:
        raw["quality_score"] = 0.0
    else:
        quality["date"] = pd.to_datetime(quality["date"])
        quality["instrument"] = quality["instrument"].astype(str)
        for col in ["quality_score", "liquidity_quality"]:
            quality[col] = pd.to_numeric(quality[col], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)
        quality["quality_score"] = quality["quality_score"] + quality["liquidity_quality"]
        raw = raw.merge(quality[["date", "instrument", "quality_score"]], how="left", on=["date", "instrument"])
        raw["quality_score"] = raw["quality_score"].fillna(0.0)

    exposure = dai.query(
        f"SELECT date, instrument, industry_level1_code FROM {exposure_table}",
        filters={"date": [query_start, query_end]},
        compression=True,
    ).df()
    if exposure.empty:
        raw["industry_level1_code"] = "UNKNOWN"
    else:
        exposure["date"] = pd.to_datetime(exposure["date"])
        exposure["instrument"] = exposure["instrument"].astype(str)
        exposure["industry_level1_code"] = exposure["industry_level1_code"].astype(object).fillna("UNKNOWN").astype(str)
        raw = raw.merge(exposure, how="left", on=["date", "instrument"])
        raw["industry_level1_code"] = raw["industry_level1_code"].astype(object).fillna("UNKNOWN").astype(str)

    raw["F01"] = raw["pressure5_mean"] - 0.60 * raw["extension"] - 0.15 * raw["range_ratio"]
    raw["F02"] = raw["micro_gap"] - 0.50 * raw["extension"] - 0.10 * raw["close_range"]
    stress = np.log1p(
        np.abs(raw["range_ratio"] / (raw["amount_sum"] / 100000000.0).replace(0.0, np.nan))
    ).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    raw["F03"] = -stress + 0.20 * np.log1p(raw["depth3_mean"].clip(lower=0.0)) - 0.20 * raw["extension"]
    consistency = raw["pressure3_abs_mean"] / (raw["pressure3_std"] + raw["pressure3_abs_mean"]).replace(0.0, np.nan)
    raw["F08"] = raw["pressure3_mean"] * consistency.fillna(0.0) - 0.35 * raw["extension"] - 0.10 * raw["close_range"]
    micro_resilience = 0.25 * np.log1p(raw["depth3_mean"].clip(lower=0.0)) - 3.00 * raw["close_range"] - 0.50 * raw["extension"].abs()
    quality_bounded = raw["quality_score"] / (1.0 + raw["quality_score"].abs())
    micro_bounded = micro_resilience / (1.0 + micro_resilience.abs())
    raw["F14"] = quality_bounded * micro_bounded
    bid_near_far = raw["bid_depth3_mean"] / (raw["bid_depth45_mean"] + 1.0)
    ask_near_far = raw["ask_depth3_mean"] / (raw["ask_depth45_mean"] + 1.0)
    raw["depth_slope_asymmetry"] = np.log1p(bid_near_far.clip(lower=0.0)) - np.log1p(ask_near_far.clip(lower=0.0))
    raw["spread_tightness"] = -raw["spread_ratio_mean"]
    raw["depth_resilience"] = np.log1p(raw["depth3_mean"].clip(lower=0.0)) - 4.0 * raw["range_ratio"]
    intraday_return = raw["close"] / raw["open"].replace(0.0, np.nan) - 1.0
    opening_gap = raw["open"] / raw["pre_close"].replace(0.0, np.nan) - 1.0
    raw["trade_exhaustion"] = np.log1p(raw["deal_number_sum"].clip(lower=0.0)) / (intraday_return.abs() + 0.005)
    raw["average_trade_size"] = np.log1p((raw["volume_sum"] / (raw["deal_number_sum"] + 1.0)).clip(lower=0.0))
    raw["intraday_reversal"] = -intraday_return
    raw["close_location"] = (
        (raw["close"] - raw["day_low"]) / (raw["day_high"] - raw["day_low"]).replace(0.0, np.nan) - 0.5
    )
    raw["opening_gap_reversal"] = -opening_gap
    raw["opening_absorption"] = -opening_gap * intraday_return
    raw["closing_pressure_shift"] = raw["closing_pressure"] - raw["opening_pressure"]
    raw["pressure_instability"] = raw["pressure3_std"]
    raw["amount_concentration"] = raw["amount_std"] / (raw["amount_mean"] + 1.0)
    raw["profitability_quality"] = raw["quality_score"]

    raw = raw.sort_values(["instrument", "date"]).reset_index(drop=True)
    grouped = raw.groupby("instrument", group_keys=False)
    raw["short_reversal_5"] = -(raw["close"] / grouped["close"].shift(5).replace(0.0, np.nan) - 1.0)
    raw["medium_momentum_20"] = raw["close"] / grouped["close"].shift(20).replace(0.0, np.nan) - 1.0
    one_day_return = raw["close"] / raw["pre_close"].replace(0.0, np.nan) - 1.0
    raw["realized_volatility_20"] = one_day_return.groupby(raw["instrument"]).transform(
        lambda series: series.rolling(20, min_periods=10).std(ddof=0)
    )
    rolling_amount = raw.groupby("instrument", group_keys=False)["amount_sum"].transform(
        lambda series: series.rolling(20, min_periods=10).median()
    )
    raw["amount_shock_20"] = raw["amount_sum"] / rolling_amount.replace(0.0, np.nan) - 1.0
    daily_illiquidity = one_day_return.abs() / (raw["amount_sum"].abs() + 1.0)
    raw["amihud_20"] = daily_illiquidity.groupby(raw["instrument"]).transform(
        lambda series: series.rolling(20, min_periods=10).mean()
    )
    raw["range_mean_5d"] = grouped["range_ratio"].transform(
        lambda series: series.rolling(5, min_periods=3).mean()
    )
    rolling_low_3d = grouped["close"].transform(lambda series: series.rolling(3, min_periods=3).min())
    raw["price_to_low_3d"] = raw["close"] / rolling_low_3d.replace(0.0, np.nan)
    raw["amount_mean_3d"] = grouped["amount_sum"].transform(
        lambda series: series.rolling(3, min_periods=3).mean()
    )
    raw["illiq_3d"] = daily_illiquidity.groupby(raw["instrument"]).transform(
        lambda series: series.rolling(3, min_periods=3).mean()
    )
    raw["order_count_cv"] = raw["order_count_std"] / (raw["order_count_mean"] + 1.0)

    residual_inputs = [
        "fragmentation",
        "frag_volatility",
        "pressure3_mean",
        "pressure5_mean",
        "extension",
        "range_ratio",
        "order_count_cv",
    ]
    ranked_residual_inputs = raw[["date", "instrument"]].copy()
    for col in residual_inputs:
        ranked_residual_inputs[col] = raw.groupby("date", group_keys=False)[col].transform(rank_z)

    def fragmentation_residual(group):
        y = group["fragmentation"].to_numpy(dtype=float)
        controls = group[["pressure3_mean", "pressure5_mean", "extension", "range_ratio"]].to_numpy(dtype=float)
        valid = np.isfinite(y) & np.isfinite(controls).all(axis=1)
        residual = np.zeros(len(group), dtype=float)
        if valid.sum() >= 20:
            design = np.column_stack([np.ones(valid.sum()), controls[valid]])
            beta = np.linalg.lstsq(design, y[valid], rcond=None)[0]
            residual[valid] = y[valid] - design.dot(beta)
        return pd.Series(residual, index=group.index)

    ranked_residual_inputs["fragmentation_residual"] = ranked_residual_inputs.groupby(
        "date", group_keys=False
    ).apply(fragmentation_residual)
    raw["F17"] = (
        ranked_residual_inputs["fragmentation_residual"]
        - 0.18 * ranked_residual_inputs["frag_volatility"]
        + 0.12 * ranked_residual_inputs["order_count_cv"]
    )

    pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={"date": [query_start, query_end]},
        compression=True,
    ).df()
    pool["date"] = pd.to_datetime(pool["date"])
    pool["instrument"] = pool["instrument"].astype(str)
    panel = pd.merge(raw, pool, how="inner", on=["date", "instrument"])
    panel["industry_level1_code"] = panel["industry_level1_code"].astype(object).fillna("UNKNOWN").astype(str)

    if "spread_tightness" in panel and "pressure_instability" in panel:
        panel["spread_pressure_quality"] = panel["spread_tightness"] - panel["pressure_instability"].abs()
    if "closing_pressure_shift" in panel and "pressure_instability" in panel:
        panel["closing_pressure_stability"] = panel["closing_pressure_shift"] - panel["pressure_instability"].abs()
    if "opening_absorption" in panel and "depth_resilience" in panel:
        panel["opening_depth_recovery"] = panel["opening_absorption"] + panel["depth_resilience"]
    if all(col in panel for col in ["pressure_instability", "F01", "F08"]):
        panel["style_residual_pressure"] = panel["pressure_instability"] - 0.5 * panel["F01"] - 0.5 * panel["F08"]
    if "depth_resilience" in panel and "realized_volatility_20" in panel:
        panel["depth_recovery_after_range"] = panel["depth_resilience"] - panel["realized_volatility_20"].abs()
    if "amihud_20" in panel and "realized_volatility_20" in panel:
        panel["liquidity_vol_balance"] = -panel["amihud_20"] - panel["realized_volatility_20"].abs()
    if "F14" in panel and "fundamental_value" in panel:
        panel["quality_value_micro"] = 0.7 * panel["F14"] + 0.3 * panel["fundamental_value"]

    all_feature_cols = feature_cols + expanded_cols + rawpool_cols
    for col in expanded_cols:
        if col not in panel:
            panel[col] = 0.0
    for col in rawpool_cols:
        if col not in panel:
            panel[col] = 0.0

    for col in all_feature_cols:
        values = pd.to_numeric(panel[col], errors="coerce").replace([np.inf, -np.inf], np.nan)
        winsorized = values.groupby(panel["date"], group_keys=False).transform(winsorize_cross_section)
        industry_fill = winsorized.groupby([panel["date"], panel["industry_level1_code"]], group_keys=False).transform("median")
        values = winsorized.fillna(industry_fill)
        daily_fill = values.groupby(panel["date"], group_keys=False).transform("median")
        values = values.fillna(daily_fill).fillna(0.0)
        industry_center = values.groupby([panel["date"], panel["industry_level1_code"]], group_keys=False).transform("median")
        panel[col] = values - industry_center
        panel[col] = panel.groupby("date", group_keys=False)[col].transform(rank_z)
        panel[col] = panel[col].replace([np.inf, -np.inf], np.nan).fillna(0.0)

    factor = panel[["date", "instrument"]].copy()
    factor["factor"] = 0.0
    for name, weight in final_weights.items():
        if name in panel:
            factor["factor"] = factor["factor"] + float(weight) * panel[name]
    factor["factor"] = factor.groupby("date", group_keys=False)["factor"].transform(rank_z)
    factor["factor"] = factor["factor"].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    factor = factor.sort_values(["instrument", "date"]).reset_index(drop=True)
    factor["factor"] = factor.groupby("instrument", group_keys=False)["factor"].transform(
        lambda series: series.rolling(4, min_periods=2).mean()
    )
    factor["factor"] = factor.groupby("date", group_keys=False)["factor"].transform(rank_z)
    factor["factor"] = factor["factor"].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    factor = factor[(factor["date"] >= start_ts.normalize()) & (factor["date"] <= end_ts)]

    submit_pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={"date": [start_date, end_date]},
        compression=True,
    ).df()
    submit_pool["date"] = pd.to_datetime(submit_pool["date"])
    submit_pool["instrument"] = submit_pool["instrument"].astype(str)

    result = pd.merge(submit_pool, factor[["date", "instrument", "factor"]], how="left", on=["date", "instrument"])
    result["factor"] = pd.to_numeric(result["factor"], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return result[["date", "instrument", "factor"]].reset_index(drop=True)
